# DML aplicado a análisis de datos
**Módulo 1 – Unidad 3** · *Data Science II: Machine Learning para la Ciencia de Datos* (Coderhouse)

Práctica del esquema propuesto en la unidad: usar `INSERT`, `UPDATE`, `DELETE` y `SELECT` para preparar un dataset listo para Machine Learning, integrando SQL con Python (sqlite3, pandas y SQLAlchemy).

Autor: Matías Rodríguez

## 1. Preparación (imports y configuración)

In [1]:
import sqlite3
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text

# Semilla fija para que los datos simulados sean reproducibles
np.random.seed(42)

# Base SQLite en un archivo local (se recrea en cada ejecución)
DB_PATH = Path("features.db")
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
print("Base creada:", DB_PATH.name)

Base creada: features.db


## 2. Crear la tabla `features` e insertar datos simulados (bulk insert)

Se simulan 50.000 registros con dos features numéricas, una categoría y una etiqueta binaria. Se introducen a propósito nulos y outliers para luego limpiarlos con DML.

In [2]:
cur.execute("""
CREATE TABLE features (
    id        INTEGER PRIMARY KEY,
    categoria TEXT,
    feature1  REAL,
    feature2  REAL,
    label     INTEGER
)
""")

N = 50_000
categorias = np.random.choice(["A", "B", "C", "D"], size=N)
feature1 = np.random.normal(50, 10, N)
feature2 = np.random.uniform(0, 1, N)
label = (feature1 + feature2 * 20 + np.random.normal(0, 5, N) > 60).astype(int)

# Ensuciamos los datos: outliers, nulos y etiquetas faltantes
idx_outliers = np.random.choice(N, 200, replace=False)
feature1[idx_outliers] = feature1[idx_outliers] * 10

registros = []
for i in range(N):
    f1 = None if i % 97 == 0 else float(feature1[i])
    f2 = None if i % 89 == 0 else float(feature2[i])
    lb = None if i % 251 == 0 else int(label[i])
    registros.append((i + 1, str(categorias[i]), f1, f2, lb))

# INSERT masivo con executemany (mucho más rápido que fila por fila)
cur.executemany(
    "INSERT INTO features (id, categoria, feature1, feature2, label) VALUES (?, ?, ?, ?, ?)",
    registros,
)
conn.commit()
print("Registros insertados:", cur.execute("SELECT COUNT(*) FROM features").fetchone()[0])

Registros insertados: 50000


## 3. Consultas exploratorias con pandas (`SELECT`)

In [3]:
df = pd.read_sql_query("SELECT * FROM features", conn)
df.head()

,id,categoria,feature1,feature2,label
0,1,C,NaN,NaN,NaN
1,2,D,50.933652,0.401747,0.0
2,3,A,43.405731,0.341812,1.0
3,4,C,50.040013,0.766517,1.0
4,5,C,42.856134,0.809105,0.0


In [4]:
# Calidad de datos: nulos por columna
pd.read_sql_query("""
SELECT
    SUM(CASE WHEN feature1 IS NULL THEN 1 ELSE 0 END) AS nulos_feature1,
    SUM(CASE WHEN feature2 IS NULL THEN 1 ELSE 0 END) AS nulos_feature2,
    SUM(CASE WHEN label    IS NULL THEN 1 ELSE 0 END) AS nulos_label,
    MAX(feature1) AS max_feature1,
    AVG(feature1) AS avg_feature1
FROM features
""", conn)

,nulos_feature1,nulos_feature2,nulos_label,max_feature1,avg_feature1
0,516,562,200,818.343392,51.835396


In [5]:
# Distribución de la etiqueta por categoría (dataset agregado para análisis)
pd.read_sql_query("""
SELECT categoria,
       COUNT(*)            AS registros,
       ROUND(AVG(label), 3) AS tasa_positivos
FROM features
WHERE label IS NOT NULL
GROUP BY categoria
ORDER BY categoria
""", conn)

,categoria,registros,tasa_positivos
0,A,12566,0.504
1,B,12383,0.501
2,C,12389,0.503
3,D,12462,0.499


## 4. `UPDATE`: capear outliers y crear `feature1_scaled`

1. Se capean los outliers de `feature1` al percentil 99.
2. Se agrega la columna `feature1_scaled`, normalizada dividiendo por el máximo (queda entre 0 y 1).

In [6]:
p99 = float(np.nanpercentile(pd.read_sql_query("SELECT feature1 FROM features", conn)["feature1"], 99))
print(f"Percentil 99 de feature1: {p99:.2f}")

cur.execute("UPDATE features SET feature1 = ? WHERE feature1 > ?", (p99, p99))
print("Outliers capeados:", cur.rowcount)

cur.execute("ALTER TABLE features ADD COLUMN feature1_scaled REAL")
cur.execute("""
UPDATE features
SET feature1_scaled = feature1 / (SELECT MAX(feature1) FROM features)
WHERE feature1 IS NOT NULL
""")
conn.commit()

pd.read_sql_query(
    "SELECT MIN(feature1_scaled) AS min_scaled, MAX(feature1_scaled) AS max_scaled, MAX(feature1) AS max_feature1 FROM features",
    conn,
)

Percentil 99 de feature1: 74.92
Outliers capeados: 495


,min_scaled,max_scaled,max_feature1
0,0.071325,1.0,74.924377


## 5. `DELETE`: eliminar registros irreparables

Se eliminan las filas sin `label` (no sirven para entrenar) y las que no tienen **ni** `feature1` **ni** `feature2`.

In [7]:
antes = cur.execute("SELECT COUNT(*) FROM features").fetchone()[0]

cur.execute("""
DELETE FROM features
WHERE label IS NULL
   OR (feature1 IS NULL AND feature2 IS NULL)
""")
conn.commit()

despues = cur.execute("SELECT COUNT(*) FROM features").fetchone()[0]
print(f"Antes: {antes} | Eliminados: {antes - despues} | Después: {despues}")

Antes: 50000 | Eliminados: 205 | Después: 49795


## 6. `SELECT` optimizado: agregación sin índice vs. con índice

Se crea un índice *cubriente* sobre `(categoria, feature1, label)`: SQLite filtra por categoría y calcula los promedios leyendo solo el índice, sin recorrer la tabla completa.

In [8]:
consulta = """
SELECT categoria, AVG(feature1) AS avg_f1, AVG(label) AS tasa_positivos
FROM features
WHERE categoria = 'B'
GROUP BY categoria
"""

def medir(sql, repeticiones=50):
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        cur.execute(sql).fetchall()
    return (time.perf_counter() - inicio) / repeticiones * 1000

t_sin = medir(consulta)
cur.execute("CREATE INDEX idx_features_cat_f1_label ON features(categoria, feature1, label)")
t_con = medir(consulta)

print(f"Sin índice: {t_sin:.2f} ms | Con índice: {t_con:.2f} ms")
print("Plan de ejecución:", cur.execute("EXPLAIN QUERY PLAN " + consulta).fetchall())

Sin índice: 3.53 ms | Con índice: 1.53 ms
Plan de ejecución: [(6, 0, 0, 'SEARCH features USING COVERING INDEX idx_features_cat_f1_label (categoria=?)')]


## 7. SQLAlchemy: insertar un batch y `to_sql` desde pandas

In [9]:
engine = create_engine(f"sqlite:///{DB_PATH}")

# Inserción de un pequeño batch con parámetros nombrados
nuevos = [
    {"id": 60001, "categoria": "A", "feature1": 48.2, "feature2": 0.31, "label": 0},
    {"id": 60002, "categoria": "C", "feature1": 61.7, "feature2": 0.85, "label": 1},
    {"id": 60003, "categoria": "D", "feature1": 55.0, "feature2": 0.47, "label": 1},
]
with engine.begin() as con:
    con.execute(
        text("INSERT INTO features (id, categoria, feature1, feature2, label) "
             "VALUES (:id, :categoria, :feature1, :feature2, :label)"),
        nuevos,
    )

# Dataset final listo para ML, guardado como nueva tabla con to_sql
conn.close()
dataset_ml = pd.read_sql_query(
    "SELECT id, categoria, feature1_scaled, feature2, label FROM features "
    "WHERE feature1_scaled IS NOT NULL AND feature2 IS NOT NULL",
    engine,
)
dataset_ml.to_sql("dataset_ml", engine, if_exists="replace", index=False)

with engine.connect() as con:
    total = con.execute(text("SELECT COUNT(*) FROM dataset_ml")).scalar()
print("Filas en dataset_ml:", total)
dataset_ml.describe()

Filas en dataset_ml: 48733


,id,feature1_scaled,feature2,label
count,48733.000000,48733.000000,48733.000000,48733.000000
mean,25001.127511,0.668452,0.498876,0.501775
std,14433.920816,0.134209,0.288972,0.500002
min,2.000000,0.071325,0.000028,0.000000
25%,12501.000000,0.576882,0.246582,0.000000
50%,25000.000000,0.668694,0.499288,1.000000
75%,37501.000000,0.759412,0.748406,1.000000
max,50000.000000,1.000000,0.999992,1.000000


## Conclusiones

- `INSERT` con `executemany` permite cargar decenas de miles de filas en segundos.
- `UPDATE` sirve para limpiar (capear outliers) y crear features derivadas (`feature1_scaled`).
- `DELETE` elimina registros que no aportan al entrenamiento (sin etiqueta o sin features).
- Un índice cubriente sobre las columnas filtradas y agregadas reduce el tiempo de la consulta (se compara en la sección 6).
- SQLAlchemy + `to_sql` integran SQL y pandas en un pipeline reproducible.